# Part 3 — MetaDrive (realistic sim + velocity action)

Ports the validated NeSy pipeline to **MetaDrive**: a continuous **velocity `(v, ω)`** action (Lab-1 `cmd_vel`), a **CBF** safety filter (Lab 5) + **velocity obstacles** (Lab 4) on the command, the **same `predicates()`** via a MetaDrive observation adapter, and **intersection rules** (2022 paper). Saves the **MetaDrive checkpoint + `part3_metadrive.mp4`** to Drive.

> ⚠️ **MetaDrive needs Python ≤ 3.11; Colab now defaults to 3.12, where MetaDrive will not install.** Training/eval cells **skip automatically** if MetaDrive is absent; the **CBF↔shield agreement check still runs**. To actually run Part 3, use the `condacolab` (Python 3.10) block in the install cell.

## 1. Setup + load the Part-2 NeSy checkpoint from Drive

In [ ]:
import os, warnings, logging
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message=r'.*Gym has been unmaintained.*')
logging.getLogger('gym').setLevel(logging.ERROR)

from google.colab import drive
drive.mount('/content/drive')

TOKEN = None
try:
    from google.colab import userdata
    TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    TOKEN = None
if not TOKEN:
    import getpass
    TOKEN = getpass.getpass('GitHub token (input hidden): ')
assert TOKEN, 'No token provided.'
os.environ['GITHUB_TOKEN'] = TOKEN
print('Token loaded (not displayed).')

In [ ]:
%%bash
set -e
GH_USER=silvergjeka22
REPO=nesy-highway-driving
curl -fsSL -H "Authorization: token ${GITHUB_TOKEN}" \
  "https://raw.githubusercontent.com/${GH_USER}/${REPO}/main/bash/setup_colab.sh" \
  -o /content/setup_colab.sh
bash /content/setup_colab.sh

# Keep NumPy in the SciPy-compatible range Colab ships (no -U: do not bump the
# stock build). <2.0 breaks SciPy ("dtype size changed"); >=2.3 breaks NumPy's
# own C-API ("_blas_supports_fpe").
pip install -q "numpy>=2.0,<2.3"


In [ ]:
# MetaDrive install (Part 3 ONLY). The CBF check + the rest of your project do not
# need it. ROOT CAUSE if it fails: MetaDrive's modern releases do not support
# Python 3.12 (Colab default), so pip backtracks to an ancient 0.2.6.0 pinning
# gym==0.19.0, which can't build on 3.12. We pin >=0.4 so that broken fallback is
# avoided; on 3.12 it simply reports no compatible build and we skip Part 3.
import sys
print('Python:', sys.version.split()[0])
!pip install -q "setuptools<66" "wheel<0.41" cython
!pip install --upgrade "metadrive-simulator>=0.4" || echo ">> No MetaDrive>=0.4 build for this Python; skip Part 3 (or use condacolab below)."
import importlib.util
MD = importlib.util.find_spec('metadrive') is not None
print('MetaDrive installed:', MD)

# --- OPTIONAL: Python 3.10 runtime so MetaDrive builds (RESTARTS kernel) ---
# !pip install -q condacolab
# import condacolab; condacolab.install()   # restarts into Python 3.10; re-run from cell 1

In [ ]:
# --- NumPy/SciPy self-heal. Colab's SciPy is built for its stock NumPy (2.0.x);
#     a newer NumPy (>=2.3) breaks the C-API ("_blas_supports_fpe"), and an older
#     one breaks the wheels ("dtype size changed"). If the numeric stack can't
#     import, force-reinstall a compatible NumPy and restart the runtime ONCE
#     (sentinel-guarded, no loop).
import os, sys, subprocess
def _stack_ok():
    try:
        import numpy as _np
        _ = _np.random.Generator
        import scipy.special  # the import chain highway-env triggers; fails on a bad NumPy
        return True
    except Exception:
        return False
if not _stack_ok() and not os.path.exists('/content/.np_repaired'):
    open('/content/.np_repaired', 'w').close()
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall',
                    '--no-cache-dir', 'numpy>=2.0,<2.3'], check=False)
    print('Repaired the NumPy/SciPy stack; restarting the runtime once. '
          'After it restarts, run Runtime > Run all again.')
    sys.stdout.flush()
    os.kill(os.getpid(), 9)
# ---------------------------------------------------------------------------

import sys, json
sys.path.insert(0, '/content/nesy-highway-driving')
%cd /content/nesy-highway-driving
from utils import load_config, drive_path, save_json
cfg = load_config('configs/highway.yaml')
from pyvirtualdisplay import Display
_display = Display(visible=0, size=(900, 480)); _display.start()
import importlib.util
MD = importlib.util.find_spec('metadrive') is not None
print('MetaDrive available:', MD)

## 2. CBF ↔ discrete shield agree (one rule, three encodings) — runs without MetaDrive
The CBF filter (Lab 5) projects a desired `(v, ω)` to a safe command; it should intervene exactly when the discrete shield would block `FASTER` on an unsafe leader gap.

In [ ]:
from labs.lab5_cbf import cbf_filter, barrier_h
from nesy.roadmap import predicates
scene = {'ego': {'x':0,'y':0,'vx':10,'vy':0,'v':10,'lane':0,'on_road':True},
         'others': [{'x':8,'y':0,'vx':4,'vy':0,'v':4,'lane':0}], 'lanes_count': 3}
preds = predicates(scene, cfg)
(v_safe, w_safe), intervened = cbf_filter(v_cmd=10.0, omega_cmd=0.0, scene=scene, cfg=cfg)
print('h(x)=', barrier_h(scene, cfg), '| shield too_close=', preds['too_close'],
      '| CBF intervened=', intervened, '| v_safe=', round(v_safe, 2))

## 3. Train PPO with a continuous (v, ω) head on MetaDrive
The discrete meta-actions stay the symbolic vocabulary (the FSM shield still reasons over manoeuvres, translated to `(v, ω)` by `labs/lab1_cmd_vel.py`); only the actuator changes.

In [ ]:
if not MD:
    md_model = None
    print('MetaDrive not installed (Python 3.12) — skipping Part 3 training.')
else:
    from agents.baselines import train_ppo_md
    md_model = train_ppo_md(cfg)   # -> checkpoints/part3_metadrive.zip

## 4. Evaluate on MetaDrive + save `part3_metadrive.mp4`
Same harness, MetaDrive env builder + observation adapter; predicates / violation monitor reused unchanged. Intersection predicates (2022 paper) are stubbed in `nesy.roadmap.intersection_predicates` pending the MetaDrive intersection API.

In [ ]:
if md_model is None:
    print('Skipping Part 3 evaluation (MetaDrive not installed).')
else:
    from eval.evaluate import evaluate, record_video
    from envs.metadrive_factory import make_env_md, read_scene_md
    md_env_fn = lambda c, render: make_env_md(c, render=render)
    md_metrics = evaluate(md_model, cfg, seeds=cfg['race']['seeds'],
                          count_violations=True, env_fn=md_env_fn, scene_fn=read_scene_md)
    save_json(md_metrics, drive_path(cfg, 'metrics', 'p3_metadrive.json'))
    print('MetaDrive metrics:', md_metrics['summary'])
    video = record_video(md_model, cfg, drive_path(cfg, 'videos', 'part3_metadrive.mp4'),
                         env_fn=md_env_fn, scene_fn=read_scene_md)
    print('Saved: checkpoints/part3_metadrive.zip  +  %s' % video)

**Exit criterion.** A MetaDrive checkpoint on Drive, the metric table (incl. CBF agreement), and **`part3_metadrive.mp4`**. If MetaDrive could not install on this Python, run Part 3 under the `condacolab` (3.10) runtime — Part 4 (the race) does not need it.